# 06 - Risultati Preliminari (Validation Set)
Generazione delle tabelle di metriche (MAE, MAPE, RMSE) e analisi Clarke Error Grid per i modelli valutati sul validation set (Sezione 3.4.5 della tesi).

In [ ]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)
import matplotlib.pyplot as plt
from lib.clarke_error_grid import clarke_error_grid_analysis

## Configurazione

In [ ]:
OUTPUT_PATH = "outputs/val_set"
SCORES_PATH = "scores/val_set"
PLOTS_PATH = "plots/val_set"

HYPO = 70.0
HYPER = 180.0

os.makedirs(SCORES_PATH, exist_ok=True)
os.makedirs(PLOTS_PATH, exist_ok=True)

print(f"Output path: {OUTPUT_PATH}")
print(f"Scores path: {SCORES_PATH}")
print(f"Plots path: {PLOTS_PATH}")

## Metriche statistiche
Calcolo di MAE, MAPE e RMSE come media (deviazione standard) dei valori per paziente, sia cumulativamente che suddivisi per condizione glicemica (ipo/normo/iper).

In [ ]:
cumulative_results = []
condition_results = []

for file in sorted(os.listdir(OUTPUT_PATH)):
    if "output" not in file:
        continue

    df = pd.read_csv(f"{OUTPUT_PATH}/{file}")
    df["bgClass"] = df["target"].apply(
        lambda x: "Hypo" if x < HYPO else ("Hyper" if x > HYPER else "Normal")
    )
    model_name = file[:-11].upper()

    # Cumulative metrics
    maes, mapes, rmses = [], [], []
    for subject in df["Patient_ID"].unique():
        x = df[df["Patient_ID"] == subject]
        maes.append(mean_absolute_error(x["target"], x["y_pred"]))
        mapes.append(mean_absolute_percentage_error(x["target"], x["y_pred"]) * 100)
        rmses.append(root_mean_squared_error(x["target"], x["y_pred"]))

    cumulative_results.append([
        model_name,
        f"{np.mean(maes):.2f}({np.std(maes):.2f})",
        f"{np.mean(mapes):.2f}({np.std(mapes):.2f})",
        f"{np.mean(rmses):.2f}({np.std(rmses):.2f})",
    ])

    # Per-condition metrics
    for condition in ["Normal", "Hyper", "Hypo"]:
        cond_df = df[df["bgClass"] == condition]
        maes, mapes, rmses = [], [], []
        for subject in df["Patient_ID"].unique():
            x = cond_df[cond_df["Patient_ID"] == subject]
            if x.empty:
                continue
            maes.append(mean_absolute_error(x["target"], x["y_pred"]))
            mapes.append(mean_absolute_percentage_error(x["target"], x["y_pred"]) * 100)
            rmses.append(root_mean_squared_error(x["target"], x["y_pred"]))

        condition_results.append([
            model_name, condition,
            f"{np.mean(maes):.2f}({np.std(maes):.2f})",
            f"{np.mean(mapes):.2f}({np.std(mapes):.2f})",
            f"{np.mean(rmses):.2f}({np.std(rmses):.2f})",
        ])

cumulative_df = pd.DataFrame(cumulative_results, columns=["Model", "MAE", "MAPE", "RMSE"])
cumulative_df.to_csv(f"{SCORES_PATH}/cumulative_results.csv", index=False)
print("Cumulative Results:")
print(cumulative_df)

condition_df = pd.DataFrame(condition_results, columns=["Model", "Condition", "MAE", "MAPE", "RMSE"])
condition_pivot = condition_df.pivot(index="Model", columns="Condition", values=["MAE", "MAPE", "RMSE"])
condition_pivot.columns = [f"{m} ({c})" for m, c in condition_pivot.columns]
condition_pivot.reset_index(inplace=True)
condition_pivot.to_csv(f"{SCORES_PATH}/condition_results.csv", index=False)
print("\nCondition Results:")
print(condition_pivot)

## Clarke Error Grid
La Clarke Error Grid classifica le predizioni in zone cliniche (A-E). Le zone A+B sono clinicamente accettabili, le zone D+E rappresentano errori potenzialmente pericolosi.

In [ ]:
output_files = sorted([f for f in os.listdir(OUTPUT_PATH) if f.endswith("_output.csv")])
ceg_results = []

for file in output_files:
    model_name = file[:-11].upper()
    df = pd.read_csv(f"{OUTPUT_PATH}/{file}")

    plot_path = f"{PLOTS_PATH}/ceg_{model_name.lower()}.png"
    stats = clarke_error_grid_analysis(
        df["target"].values, df["y_pred"].values, model_name, save_path=plot_path
    )

    print(f"\n{model_name}:")
    for zone, pct in stats["zone_percentages"].items():
        print(f"  Zone {zone}: {pct:.2f}%")
    print(f"  Clinically Acceptable (A+B): {stats['clinically_acceptable']:.2f}%")

    ceg_results.append({
        "Model": model_name,
        **{f"Zone_{z}": f"{p:.2f}%" for z, p in stats["zone_percentages"].items()},
        "Acceptable_AB": f"{stats['clinically_acceptable']:.2f}%",
        "Dangerous_DE": f"{stats['clinically_dangerous']:.2f}%",
    })

ceg_df = pd.DataFrame(ceg_results)
ceg_df.to_csv(f"{SCORES_PATH}/ceg_zones_results.csv", index=False)
print("\nCEG Summary:")
print(ceg_df)